In [0]:
df = spark.read.table("workspace.tfl.silver_journeys")

In [0]:
# create temporary view for SQL queries
df.createOrReplaceTempView("silver")

In [0]:
daily_hires = spark.sql("""
    SELECT 
        start_time::date AS date,
        COUNT(*) AS hires,
        ROUND(AVG(duration_minutes), 2) AS avg_duration_minutes
    FROM silver
    GROUP BY start_time::date
    ORDER BY date;
    """)
daily_hires.display()
daily_hires.write.mode("overwrite").saveAsTable("workspace.tfl.gold_daily_hires")

spark.sql("COMMENT ON TABLE workspace.tfl.gold_daily_hires IS 'Number of hires and average duration of trips per day'")
spark.sql("COMMENT ON COLUMN workspace.tfl.gold_daily_hires.hires IS 'Total count of hires per day'")

In [0]:
# count number of hires starting and ending per each station id per month

station_activity = spark.sql("""
    WITH starts AS (
    SELECT 
        start_station_id AS station_id,
        month(start_time) AS month,
        COUNT(*) AS starting_hires
    FROM silver
    GROUP BY start_station_id, month(start_time)
    ),
    ends AS (
        SELECT 
            end_station_id AS station_id,
            month(end_time) AS month,
            COUNT(*) AS ending_hires
        FROM silver
        GROUP BY end_station_id, month(end_time)
    )
    SELECT 
        COALESCE(s.station_id, e.station_id) AS station_id,
        COALESCE(s.month, e.month) AS month,
        COALESCE(s.starting_hires, 0) AS starting_hires,
        COALESCE(e.ending_hires, 0) AS ending_hires
    FROM starts s
    FULL OUTER JOIN ends e
        ON s.station_id = e.station_id
        AND s.month = e.month
    ORDER BY station_id, month;
    """)
station_activity.display()
station_activity.write.mode("overwrite").saveAsTable("workspace.tfl.gold_station_activity")

spark.sql("COMMENT ON TABLE workspace.tfl.gold_station_activity IS 'Summary of hires and returns per station per month'")

In [0]:
# top 500 most popular start and end station pairs with typical journey times

popular_routes = spark.sql("""
    SELECT 
        start_station_id,
        start_station_name,
        end_station_id, 
        end_station_name,
        COUNT(*) as hires,
        ROUND(AVG(duration_minutes), 2) as avg_duration_minutes
    FROM silver
    GROUP BY start_station_id, start_station_name, end_station_id, end_station_name
    ORDER BY hires DESC
    LIMIT 500;  
    """)

popular_routes.display()
popular_routes.write.mode("overwrite").saveAsTable("workspace.tfl.gold_popular_routes")

spark.sql("COMMENT ON TABLE workspace.tfl.gold_popular_routes IS 'Top 500 most popular start and end station pairs with typical journey times'")
spark.sql("COMMENT ON COLUMN workspace.tfl.gold_popular_routes.hires IS 'Total count of hires between given start and end stations'")


In [0]:
# find most common times of day for bike hires to start

start_hours = spark.sql("""
    SELECT       
        hour(start_time) AS start_hour,
        COUNT(*) as hires,
        ROUND(AVG(duration_minutes), 2) AS avg_duration_minutes
    FROM silver
    GROUP BY hour(start_time)
    ORDER BY start_hour;
    """)

start_hours.display()
start_hours.write.mode("overwrite").saveAsTable("workspace.tfl.gold_start_hours")

spark.sql("COMMENT ON TABLE workspace.tfl.gold_start_hours IS 'Number of hires and average duration of trips per hour of day'")

Databricks visualization. Run in Databricks to view.

Databricks visualization. Run in Databricks to view.